# Dynamic Attack Adaptation — IDS Pipeline
## Phase 1 & 2 Walkthrough Notebook
**Project:** B.Tech Sem IV Research | NMIMS — Design Experience / IROP  
**Dataset:** CICIoT2023 (University of New Brunswick)  
**Pipeline:** Network Traffic → Preprocessing → Feature Selection → [handoff to Phase 3]

---
### ⚙️ How to use this notebook
- **On your laptop**: Run cells top to bottom. Edit `SAMPLE_PER_CLASS` to keep it fast.
- **On Google Colab**: Upload your CSVs to Google Drive, mount it, and update `DATA_RAW_DIR`.

> **Note:** This notebook mirrors `run_phase1_2.py` step-by-step with explanations for each stage.

## 0. Setup

In [ ]:
# Install dependencies (run once)
# On Colab these may already be present; safe to re-run.
import subprocess, sys
pkgs = [
    'pandas', 'numpy', 'scikit-learn', 'imbalanced-learn',
    'pyarrow', 'matplotlib', 'seaborn', 'joblib', 'tqdm'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print('All packages ready.')

In [ ]:
# ---- COLAB ONLY: mount Google Drive ----
import os
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
else:
    print('Running locally — no mount needed.')

In [ ]:
# ================================================================
# CONFIGURATION — edit these paths for your environment
# ================================================================
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    # --- COLAB: adjust this path to where your CSVs are in Drive ---
    DATA_RAW_DIR = Path('/content/drive/MyDrive/CICIoT2023')
    PROJECT_ROOT = Path('/content/DYNAMIC-ATTACK-ADAPTION-IDS')
else:
    # --- LAPTOP ---
    DATA_RAW_DIR = Path(r'D:\IROP MODEL\CICIOT23')
    PROJECT_ROOT = Path(r'D:\IROP MODEL\DYNAMIC-ATTACK-ADAPTION-IDS')

# ---- Clone repo if running in Colab ----
if IN_COLAB and not PROJECT_ROOT.exists():
    import subprocess
    subprocess.run(['git', 'clone', '-b', 'phase1-2-prototype',
                    'https://github.com/SurajSahoo2006/DYNAMIC-ATTACK-ADAPTION-IDS',
                    str(PROJECT_ROOT)])

import sys
sys.path.insert(0, str(PROJECT_ROOT))

# Override config paths
import config
config.DATA_RAW_DIR   = DATA_RAW_DIR
config.TRAIN_CSV      = DATA_RAW_DIR / 'train' / 'train.csv'
config.TEST_CSV       = DATA_RAW_DIR / 'test'  / 'test.csv'
config.DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
config.ARTIFACTS_DIR      = PROJECT_ROOT / 'artifacts'
config.REPORTS_DIR        = PROJECT_ROOT / 'reports'

for d in [config.DATA_PROCESSED_DIR, config.ARTIFACTS_DIR, config.REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Sampling knob ----
# Laptop: 5_000 rows/class ≈ 170k rows total, ~2 min
# Colab T4: 50_000 rows/class ≈ 1.7M rows total, ~15 min
SAMPLE_PER_CLASS = 5_000   # ← CHANGE THIS
config.SAMPLE_PER_CLASS = SAMPLE_PER_CLASS
config.SMOTE_TARGET_COUNT = SAMPLE_PER_CLASS

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'DATA_RAW_DIR  : {DATA_RAW_DIR}')
print(f'SAMPLE_PER_CLASS: {SAMPLE_PER_CLASS}')
print(f'Train CSV exists: {config.TRAIN_CSV.exists()}')

---
## Phase 1 — Data Ingestion
**Architecture box:** *Network Traffic Capture / Honeypot Enrichment / Benchmark Datasets → Preprocessing*

### What happens here:
1. Read the CICIoT2023 CSV files in chunks (avoids OOM on a laptop).
2. Sample `SAMPLE_PER_CLASS` rows per fine-grained label (34 classes).
3. Print a validation report (row counts, missing values, label distribution).
4. Plot the class distribution bar chart.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)   # reduce log noise in notebook

from ingest.loader import load_ciciot2023

df_raw = load_ciciot2023(sample_per_class=SAMPLE_PER_CLASS)
print(f'\nLoaded shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
from ingest.validate import run_validation_report
val_summary = run_validation_report(df_raw)

In [ ]:
from IPython.display import Image
Image(str(config.REPORTS_DIR / 'class_distribution.png'))

In [ ]:
# Cowrie Honeypot module — returns empty DataFrame if no log provided
from ingest.cowrie_loader import load_cowrie_log, load_benchmark_stub
df_cowrie = load_cowrie_log()   # returns empty — no log file in this run
print(f'Cowrie rows: {len(df_cowrie)}  (0 = no log file provided, pipeline continues cleanly)')

---
## Phase 2 — Preprocessing & Feature Selection

**Architecture box:** *Preprocessing (SMOTE + Normalization + Label Encoding)*  
then → *Feature Selection (RFE — Top 19 Features)*

### Order matters — here is why:

| Step | Why this order? |
|------|----------------|
| 1. Clean | Remove bad rows before anything else |
| 2. Encode labels | Create integer labels for sklearn |
| 3. **Split first** | Prevents test data leaking into scaler/SMOTE |
| 4. Scale (train only) | Scaler never sees test data |
| 5. SMOTE (train only) | Test set reflects real-world imbalance |
| 6. RFE | Feature importance measured on balanced train set |

In [ ]:
# Step 2.1 — Clean
from preprocess.clean import clean
df_clean, clean_summary = clean(df_raw)

In [ ]:
# Step 2.2 — Encode labels
from preprocess.encode import encode_labels
df_encoded, le_34, le_8 = encode_labels(df_clean)
del df_clean   # free memory
df_encoded[['label', 'label_34', 'label_8']].drop_duplicates().sort_values('label_34').head(10)

In [ ]:
# Step 2.3 — Stratified split BEFORE scaling or SMOTE
from ingest.loader import FEATURE_COLS
from preprocess.split import split
X_train, X_test, y34_train, y34_test, y8_train, y8_test = split(df_encoded, FEATURE_COLS)
del df_encoded
print(f'Train: {X_train.shape}   Test: {X_test.shape}')

In [ ]:
# Step 2.4 — StandardScaler (fit on train only)
from preprocess.scale import scale
X_train_s, X_test_s, scaler = scale(X_train, X_test)
del X_train, X_test

In [ ]:
# Step 2.5 — SMOTE (train only)
from preprocess.balance import balance
X_train_b, y34_train_b = balance(X_train_s, y34_train)

In [ ]:
# Step 2.6 — Recursive Feature Elimination
from preprocess.select_features import select_features
selected_features, ranking_df = select_features(X_train_b, y34_train_b)

In [ ]:
Image(str(config.REPORTS_DIR / 'rfe_ranking.png'))

In [ ]:
# Step 2.7 — Save Parquet files
import pandas as pd
from config import LABEL_TO_CLASS, LABEL_34_COL, LABEL_8_COL, TRAIN_PARQUET, TEST_PARQUET

# Rebuild y8 after SMOTE (y34 was resampled)
le_34_classes = list(le_34.classes_)
y8_train_b = y34_train_b.map(
    lambda i: le_8.transform([LABEL_TO_CLASS[le_34_classes[i]]])[0]
)

train_out = pd.concat([
    X_train_b[selected_features],
    y34_train_b.rename(LABEL_34_COL),
    y8_train_b.rename(LABEL_8_COL),
], axis=1)

test_out = pd.concat([
    X_test_s[selected_features],
    y34_test.rename(LABEL_34_COL),
    y8_test.rename(LABEL_8_COL),
], axis=1)

train_out.to_parquet(config.TRAIN_PARQUET, index=False)
test_out.to_parquet(config.TEST_PARQUET, index=False)
print(f'Train saved: {train_out.shape} → {config.TRAIN_PARQUET.name}')
print(f'Test  saved: {test_out.shape}  → {config.TEST_PARQUET.name}')

---
## 🔬 Smoke Test
> **This is NOT the final classifier.** It is a quick sanity check to confirm the preprocessing pipeline produced clean, meaningful data.
> The real classifier (Phase 3: Random Forest + XGBoost + DNN ensemble) will be trained separately.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from config import RANDOM_SEED, SMOKE_TEST_TREES

X_tr = train_out[selected_features].values
y_tr = train_out[LABEL_8_COL].values
X_te = test_out[selected_features].values
y_te = test_out[LABEL_8_COL].values

clf = RandomForestClassifier(
    n_estimators=SMOKE_TEST_TREES, n_jobs=-1, random_state=RANDOM_SEED
)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

acc = accuracy_score(y_te, y_pred)
print(f'Smoke-test accuracy (8-class): {acc*100:.2f}%')
print('(Expect >90% if preprocessing is correct)')
print()
print(classification_report(y_te, y_pred, target_names=list(le_8.classes_), zero_division=0))

---
## ✅ Phase 1 & 2 Complete — Handoff to Phase 3

The following files are ready:

| File | Purpose |
|------|---------|
| `data/processed/train_phase2.parquet` | SMOTE-balanced train set (19 features) |
| `data/processed/test_phase2.parquet` | Untouched test set (19 features) |
| `artifacts/standard_scaler.pkl` | For scaling live traffic |
| `artifacts/label_encoder_34class.pkl` | Decode fine-grained predictions |
| `artifacts/label_encoder_8class.pkl` | Decode 8-class predictions |
| `artifacts/selected_features.json` | The 19 feature names |

**Next step:** Phase 3 — Ensemble Classifier (Random Forest + XGBoost + DNN)